In [0]:
import pandas as pd


df= pd.read_csv(r'/Workspace/Users/prarthanabg7@gmail.com/SQL/Github sql/customer-trends-data-analysis-Python-SQL-PowerBI/customer_shopping_behavior.csv')
df

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3895,3896,40,Female,Hoodie,Clothing,28,Virginia,L,Turquoise,Summer,4.2,No,2-Day Shipping,No,No,32,Venmo,Weekly
3896,3897,52,Female,Backpack,Accessories,49,Iowa,L,White,Spring,4.5,No,Store Pickup,No,No,41,Bank Transfer,Bi-Weekly
3897,3898,46,Female,Belt,Accessories,33,New Jersey,L,Green,Spring,2.9,No,Standard,No,No,24,Venmo,Quarterly
3898,3899,44,Female,Shoes,Footwear,77,Minnesota,S,Brown,Summer,3.8,No,Express,No,No,24,Venmo,Weekly


In [0]:
import spark 

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-4647978122712464>, line 1
----> 1 import spark 

ModuleNotFoundError: No module named 'spark'

In [0]:
df.head()

In [0]:
df.info()

In [0]:
df.describe(include='all')

In [0]:
df.isnull().sum()

In [0]:
df['Review Rating']=df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [0]:
df['Review Rating'].isnull().sum()

In [0]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ', '_')
df=df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [0]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [0]:
#create a column age_group
labels=['Young Adult','Adult','Middle-aged','Senior']
df['age_group']=pd.qcut(df['age'],q=4,labels=labels)

In [0]:
df['age_group'].head()

0    Middle-aged
1    Young Adult
2    Middle-aged
3    Young Adult
4    Middle-aged
Name: age_group, dtype: category
Categories (4, object): ['Young Adult' < 'Adult' < 'Middle-aged' < 'Senior']

In [0]:
#create column purchase_frequency_days

df['frequency_of_purchases'].unique()
frequency_mapping = {
   'Fortnightly' :14,
   'Weekly':7,
    'Annually':365,
    'Monthly':30,
    'Semi-Annually':180,
    'Quarterly':90,
    'Bi-Weekly':14,
    'Every 3 months': 90
}
df['purchase_frequency_days']=df['frequency_of_purchases'].map(frequency_mapping)


In [0]:
df['purchase_frequency_days']

0        14.0
1        14.0
2         7.0
3         7.0
4       365.0
        ...  
3895      7.0
3896     14.0
3897     90.0
3898      7.0
3899     90.0
Name: purchase_frequency_days, Length: 3900, dtype: float64

In [0]:
(df['discount_applied']==df['promo_code_used']).all()

np.True_

In [0]:
df=df.drop('promo_code_used',axis=1)

In [0]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [0]:
df.to_csv(r'/Workspace/Users/prarthanabg7@gmail.com/SQL/Github sql/customer-trends-data-analysis-Python-SQL-PowerBI/customer_shopping_behavior_clean.csv',index=False)


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   customer_id              3900 non-null   int64   
 1   age                      3900 non-null   int64   
 2   gender                   3900 non-null   object  
 3   item_purchased           3900 non-null   object  
 4   category                 3900 non-null   object  
 5   purchase_amount          3900 non-null   int64   
 6   location                 3900 non-null   object  
 7   size                     3900 non-null   object  
 8   color                    3900 non-null   object  
 9   season                   3900 non-null   object  
 10  review_rating            3863 non-null   float64 
 11  subscription_status      3900 non-null   object  
 12  shipping_type            3900 non-null   object  
 13  discount_applied         3900 non-null   object  
 14  previous

In [0]:
df['age_group'] = df['age_group'].astype(str)

In [0]:
spark_df=spark.createDataFrame(df)
database_name='customer_behavior'
spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_name}")
table_name='customer_shopping_behavior'
spark_df.write.mode('overwrite').saveAsTable(f"{database_name}.{table_name}")

In [0]:
spark_df.count()


3900